# MAD Identity Verification — Colab demo

Этот notebook запускает KYC-like pipeline для проверки личности по паспорту и selfie. Проект работает в Google Colab, потому что локальная установка PaddleOCR/DeepFace на Windows нестабильна и требует заметных ресурсов. Интерфейс запускается через Gradio, а хранение профилей может идти в Supabase или в локальный JSON fallback.

Основной поток:

```text
регистрация профиля → OCR паспорта → сверка данных → сравнение лиц → ACCEPT / REVIEW / REJECT
```

В notebook нет встроенного base64-архива, чтобы Colab не лагал. Код проекта находится в репозитории и запускается из текущей рабочей папки Colab. Notebook содержит compatibility-настройки для Colab runtime.


## 1. Установка окружения

В этой ячейке я готовлю Colab runtime: ставлю Supabase/Gradio/DeepFace, возвращаю PaddleOCR на GPU и отдельно чиню совместимость Torch для PaddleX/ModelScope. Такой путь нужен из-за конфликта CUDA/NCCL: Paddle должен использовать GPU, а Torch достаточно оставить CPU-only, потому что Torch здесь нужен только как транзитивная зависимость ModelScope.

Также здесь создаётся минимальный compatibility layer для LangChain. PaddleX ожидает старые импорты вида `langchain.docstore.document`, а современные пакеты LangChain устроены иначе. Я не даунгрейжу весь Colab-стек, а добавляю маленький shim только для нужных импортов.


In [ ]:
#@title 1. Установка зависимостей: Colab GPU, PaddleOCR, DeepFace и compatibility-слой
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# Главное отличие v4:
# 1) PaddleOCR возвращён как основной OCR.
# 2) После установки paddlepaddle-gpu чинится torch-import для PaddleX/ModelScope.
#    Это нужно из-за ошибки вида: libtorch_cuda.so: undefined symbol: ncclCommShrink.
# 3) CPU Torch ставится только для корректного импорта ModelScope. OCR всё равно использует Paddle GPU.

PADDLE_INSTALL_MODE = "gpu"  # "gpu" или "cpu"
PADDLE_CANDIDATES = [
    ("3.3.1", "cu126"),
    ("3.3.1", "cu129"),
    ("3.2.2", "cu126"),
    ("3.2.2", "cu118"),
]
PADDLE_CPU_VERSION = "3.3.0"
PADDLEOCR_VERSION = "3.3.3"
FORCE_CPU_TORCH_FOR_MODELSCOPE = True

os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")
os.environ.setdefault("DEEPFACE_HOME", "/root")

def run(cmd, check=True):
    print("\n$", " ".join(map(str, cmd)))
    return subprocess.run(cmd, check=check)

def py_install(args, check=True):
    return run([sys.executable, "-m", "pip", "install", *args], check=check)

def py_uninstall(args, check=False):
    return run([sys.executable, "-m", "pip", "uninstall", "-y", *args], check=check)

print("=== GPU check ===")
subprocess.run(["nvidia-smi"], check=False)

print("\n=== Base utilities ===")
py_install(["-q", "--upgrade", "pip", "setuptools", "wheel"], check=False)
py_install([
    "-q",
    "requests==2.32.4",
    "supabase>=2.0.0",
    "python-dotenv",
    "rapidfuzz",
    "joblib",
    "scikit-learn",
    "gradio>=4.44.0",
    "pandas",
], check=False)

print("\n=== DeepFace minimal install ===")
# DeepFace через --no-deps, чтобы не ломать Colab numpy/tensorflow/opencv.
py_uninstall(["deepface", "retina-face"], check=False)
py_install(["-q", "--no-deps", "deepface==0.0.100", "retina-face==0.0.17"], check=False)
py_install(["-q", "tf-keras", "gdown", "fire", "gunicorn", "mtcnn", "lightphe", "lightdsa", "flask", "flask-cors"], check=False)
py_install(["-q", "requests==2.32.4"], check=False)

print("\n=== PaddlePaddle install ===")
py_uninstall(["paddlepaddle", "paddlepaddle-gpu"], check=False)

if PADDLE_INSTALL_MODE == "gpu":
    installed = False
    last_rc = None
    for version, cuda_tag in PADDLE_CANDIDATES:
        index_url = f"https://www.paddlepaddle.org.cn/packages/stable/{cuda_tag}/"
        print(f"\nTrying paddlepaddle-gpu=={version} from {index_url}")
        result = py_install(["-q", f"paddlepaddle-gpu=={version}", "-i", index_url], check=False)
        last_rc = result.returncode
        if result.returncode == 0:
            installed = True
            print(f"Installed paddlepaddle-gpu=={version} ({cuda_tag})")
            break
    if not installed:
        raise RuntimeError(f"Could not install paddlepaddle-gpu. Last returncode={last_rc}")
else:
    py_install(["-q", f"paddlepaddle=={PADDLE_CPU_VERSION}", "-i", "https://www.paddlepaddle.org.cn/packages/stable/cpu/"], check=True)

print("\n=== PaddleOCR install ===")
py_install(["-q", f"paddleocr=={PADDLEOCR_VERSION}", "langchain-text-splitters>=1.0.0,<2.0.0"], check=False)

print("\n=== Torch repair for PaddleX / ModelScope ===")
# PaddleOCR 3.x imports PaddleX; PaddleX imports ModelScope; ModelScope imports torch.
# Installing Paddle GPU can replace nvidia-* packages and break Colab's preinstalled CUDA torch.
# We do not need torch GPU for this project, so CPU torch is the safest compatibility layer for ModelScope.
if FORCE_CPU_TORCH_FOR_MODELSCOPE:
    py_install([
        "-q",
        "--force-reinstall",
        "--no-cache-dir",
        "torch",
        "--index-url",
        "https://download.pytorch.org/whl/cpu",
    ], check=False)
else:
    test = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"], text=True)
    if test.returncode != 0:
        py_install([
            "-q",
            "--force-reinstall",
            "--no-cache-dir",
            "torch",
            "--index-url",
            "https://download.pytorch.org/whl/cpu",
        ], check=False)

# Fix requests again in case another package changed it.
py_install(["-q", "requests==2.32.4"], check=False)

# ------------------------------------------------------------
# PaddleOCR / PaddleX / LangChain compatibility layer
# В Colab PaddleOCR 3.x импортирует PaddleX, а PaddleX ожидает старые пути LangChain.
# Я добавляю минимальный shim, чтобы не даунгрейдить весь Colab-стек.
# ------------------------------------------------------------

import os
import sys
import site
import shutil
import subprocess
from pathlib import Path

os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
os.environ["DISABLE_MODEL_SOURCE_CHECK"] = "True"

print("=== Installing minimal LangChain core deps ===")

def run(cmd, check=True):
    print("\n$", " ".join(cmd))
    result = subprocess.run(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout[-5000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with returncode={result.returncode}: {' '.join(cmd)}")
    return result

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "langchain-core>=0.3.0,<2.0.0",
    "langchain-text-splitters>=0.3.0,<2.0.0",
])

print("\n=== Creating compatibility shim: langchain.docstore.document ===")

SHIM_ROOT = Path("/content/paddleocr_langchain_shim")
LANGCHAIN_DIR = SHIM_ROOT / "langchain"
DOCSTORE_DIR = LANGCHAIN_DIR / "docstore"

if SHIM_ROOT.exists():
    shutil.rmtree(SHIM_ROOT)

DOCSTORE_DIR.mkdir(parents=True, exist_ok=True)

# Минимальный fake langchain package.
(LANGCHAIN_DIR / "__init__.py").write_text(
    '''
"""
Minimal LangChain compatibility shim for PaddleX/PaddleOCR in Colab.

It only provides old import paths expected by PaddleX:
- langchain.docstore.document.Document
- langchain.text_splitter.RecursiveCharacterTextSplitter
"""
__version__ = "0.0.0-paddleocr-compat-shim"
'''.strip() + "\n",
    encoding="utf-8",
)

(DOCSTORE_DIR / "__init__.py").write_text(
    "from .document import Document\n",
    encoding="utf-8",
)

(DOCSTORE_DIR / "document.py").write_text(
    '''
try:
    from langchain_core.documents import Document
except Exception:
    from langchain_core.documents.base import Document

__all__ = ["Document"]
'''.strip() + "\n",
    encoding="utf-8",
)

# Старый путь, который часто ожидает PaddleX:
# from langchain.text_splitter import RecursiveCharacterTextSplitter
(LANGCHAIN_DIR / "text_splitter.py").write_text(
    '''
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except Exception:
    from langchain_text_splitters.character import RecursiveCharacterTextSplitter

__all__ = ["RecursiveCharacterTextSplitter"]
'''.strip() + "\n",
    encoding="utf-8",
)

# Дополнительный совместимый путь на всякий случай.
(LANGCHAIN_DIR / "schema.py").write_text(
    '''
try:
    from langchain_core.documents import Document
except Exception:
    from langchain_core.documents.base import Document

__all__ = ["Document"]
'''.strip() + "\n",
    encoding="utf-8",
)

print("Shim created at:", SHIM_ROOT)

print("\n=== Registering shim globally through .pth ===")

site_packages = []
for p in site.getsitepackages():
    pp = Path(p)
    if pp.exists():
        site_packages.append(pp)

# На всякий случай добавим и текущий usersite, если он существует.
try:
    user_site = Path(site.getusersitepackages())
    user_site.mkdir(parents=True, exist_ok=True)
    site_packages.append(user_site)
except Exception:
    pass

if not site_packages:
    raise RuntimeError("Could not find site-packages directory")

for sp in site_packages:
    pth = sp / "zz_paddleocr_langchain_shim.pth"
    pth.write_text(str(SHIM_ROOT) + "\n", encoding="utf-8")
    print("Wrote:", pth)

# Для текущего процесса тоже добавим путь сразу.
if str(SHIM_ROOT) not in sys.path:
    sys.path.insert(0, str(SHIM_ROOT))

# И для subprocess-ов, которые будут запущены из этого runtime.
old_pythonpath = os.environ.get("PYTHONPATH", "")
parts = [str(SHIM_ROOT)]
if old_pythonpath:
    parts.append(old_pythonpath)
os.environ["PYTHONPATH"] = os.pathsep.join(parts)

# Удаляем возможные битые/частично загруженные модули.
for name in list(sys.modules.keys()):
    if (
        name == "langchain"
        or name.startswith("langchain.")
        or name == "paddleocr"
        or name.startswith("paddleocr.")
        or name == "paddlex"
        or name.startswith("paddlex.")
    ):
        sys.modules.pop(name, None)

print("\n=== Import checks in current process ===")

from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

print("Document shim OK:", Document)
print("TextSplitter shim OK:", RecursiveCharacterTextSplitter)

print("\n=== Import checks in fresh subprocess ===")

env = os.environ.copy()
env["PYTHONPATH"] = str(SHIM_ROOT) + os.pathsep + env.get("PYTHONPATH", "")

checks = {
    "langchain_shim": """
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
print("langchain shim OK", Document, RecursiveCharacterTextSplitter)
""",
    "paddle": """
import paddle
print("paddle", paddle.__version__, "cuda", paddle.is_compiled_with_cuda())
""",
    "paddleocr": """
from paddleocr import PaddleOCR
print("paddleocr import OK")
""",
}

for name, code in checks.items():
    print(f"\n--- check: {name} ---")
    result = subprocess.run(
        [sys.executable, "-c", code],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        env=env,
    )

    if result.stdout:
        print(result.stdout[-4000:])
    if result.stderr:
        print(result.stderr[-4000:])

    if result.returncode != 0:
        raise RuntimeError(f"Import check failed: {name}")

print("\n=== Setting OCR env ===")

try:
    import paddle
    paddle_gpu_ok = bool(paddle.is_compiled_with_cuda())
except Exception:
    paddle_gpu_ok = False

os.environ["OCR_DEVICE"] = "gpu:0" if paddle_gpu_ok else "cpu"
os.environ["OCR_WORKER_TIMEOUT_SEC"] = "900"
os.environ["OCR_FAST_MODE"] = "1"
os.environ["OCR_QUALITY_PRESETS"] = "0"
os.environ["PADDLE_DET_LIMIT_TYPE"] = "max"
os.environ["PADDLE_DET_LIMIT_SIDE_LEN"] = "1600"
os.environ["PADDLE_TEXTLINE_ORIENTATION"] = "false"
os.environ["PADDLE_USE_DOC_UNWARPING"] = "false"
os.environ["PADDLE_DET_MODEL"] = "PP-OCRv5_mobile_det"
os.environ["PADDLE_REC_MODEL"] = "cyrillic_PP-OCRv5_mobile_rec"

print("OCR_DEVICE:", os.environ["OCR_DEVICE"])
print("PaddleOCR compatibility layer is ready.")

print("\n=== Import checks in fresh subprocesses ===")
checks = {
    "torch": "import torch; print('torch', torch.__version__, 'cuda_available', torch.cuda.is_available() if hasattr(torch, 'cuda') else False)",
    "paddle": "import paddle; print('paddle', paddle.__version__, 'cuda', paddle.is_compiled_with_cuda())",
    "paddleocr": "import os; os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK']='True'; from paddleocr import PaddleOCR; print('paddleocr import OK')",
    "deepface": "from deepface import DeepFace; print('deepface import OK')",
}

for name, code in checks.items():
    print(f"\n--- check: {name} ---")
    result = subprocess.run([sys.executable, "-c", code], text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print(result.stdout[-2000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
        if name in {"paddle", "paddleocr"}:
            raise RuntimeError(f"Critical import check failed: {name}")

print("\nInstall cell finished.")
print("Если эта ячейка прошла без Critical import errors, можно идти дальше без ручной локальной установки на Windows.")


## 1.5. Калибровка face threshold по CelebA

Эта ячейка опциональная. Я использую CelebA не для дообучения DeepFace, а только для оценки расстояний между positive/negative парами лиц. Для честной калибровки нужен файл identity-разметки `identity_CelebA.txt` или `list_identity_celeba.txt`; обычный partition-файл не подходит, потому что он не задаёт личности.


In [ ]:
#@title 1.5. CelebA threshold calibration до конфигурации проекта
# Эта ячейка опциональная. Она НЕ обучает DeepFace, а только оценивает cosine-distance threshold.
# Для честных positive/negative pairs нужен файл identity_CelebA.txt или list_identity_celeba.txt.

RUN_CELEBA_THRESHOLD = False #@param {type:"boolean"}
CELEBA_ROOT = "/content/drive/MyDrive/CelebA" #@param {type:"string"}
CELEBA_MODEL_NAME = "Facenet512" #@param {type:"string"}
CELEBA_DETECTOR_BACKEND = "skip" #@param ["skip", "opencv", "retinaface"]
CELEBA_MAX_IDENTITIES = 60 #@param {type:"integer"}
CELEBA_POSITIVE_PAIRS = 200 #@param {type:"integer"}
CELEBA_NEGATIVE_PAIRS = 200 #@param {type:"integer"}
CELEBA_RANDOM_SEED = 42 #@param {type:"integer"}

import json
import math
import os
import random
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")


def _cosine_distance(a, b):
    va = np.asarray(a, dtype="float32")
    vb = np.asarray(b, dtype="float32")
    denom = float(np.linalg.norm(va) * np.linalg.norm(vb))
    if denom <= 1e-12:
        return float("nan")
    return float(1.0 - np.dot(va, vb) / denom)


def _find_identity_file(root: Path) -> Path:
    candidates = [
        root / "identity_CelebA.txt",
        root / "list_identity_celeba.txt",
        root / "Anno" / "identity_CelebA.txt",
        root / "Anno" / "list_identity_celeba.txt",
        root / "Eval" / "identity_CelebA.txt",
        root / "Eval" / "list_identity_celeba.txt",
    ]
    for p in candidates:
        if p.exists():
            return p
    hits = list(root.rglob("*identity*CelebA*.txt")) + list(root.rglob("*identity*celeba*.txt"))
    if hits:
        return hits[0]
    raise FileNotFoundError("Не найден identity_CelebA.txt/list_identity_celeba.txt. list_eval_partition.txt для threshold не подходит.")


def _find_images_dir(root: Path) -> Path:
    candidates = [
        root / "img_align_celeba",
        root / "Img" / "img_align_celeba",
        root / "img_celeba",
        root / "Img" / "img_celeba",
    ]
    for p in candidates:
        if p.exists() and p.is_dir():
            return p
    for p in root.rglob("000001.jpg"):
        return p.parent
    raise FileNotFoundError("Не найдена папка изображений. Ожидаю img_align_celeba или Img/img_align_celeba внутри CelebA.")


def _load_groups(identity_path: Path, images_dir: Path, max_identities: int, seed: int):
    rng = random.Random(seed)
    groups = defaultdict(list)
    with identity_path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            image_id, person_id = parts[0], parts[1]
            p = images_dir / image_id
            if p.exists():
                groups[person_id].append(p)
    groups = {k: v for k, v in groups.items() if len(v) >= 2}
    ids = list(groups.keys())
    rng.shuffle(ids)
    ids = ids[:max_identities]
    return {k: groups[k] for k in ids}


def _build_pairs(groups, max_pos, max_neg, seed):
    rng = random.Random(seed)
    positives = []
    for imgs in groups.values():
        if len(imgs) >= 2:
            a, b = rng.sample(imgs, 2)
            positives.append((a, b, 1))
    rng.shuffle(positives)
    positives = positives[:max_pos]
    ids = list(groups.keys())
    negatives = []
    tries = 0
    while len(negatives) < max_neg and tries < max_neg * 30 and len(ids) >= 2:
        tries += 1
        ia, ib = rng.sample(ids, 2)
        negatives.append((rng.choice(groups[ia]), rng.choice(groups[ib]), 0))
    pairs = positives + negatives
    rng.shuffle(pairs)
    return pairs


def _represent_cache(paths, cache_path: Path, model_name: str, detector_backend: str):
    cache = {}
    if cache_path.exists():
        cache = json.loads(cache_path.read_text(encoding="utf-8"))
    missing = [p for p in paths if str(p) not in cache]
    if missing:
        from deepface import DeepFace
        for i, p in enumerate(missing, 1):
            try:
                result = DeepFace.represent(
                    img_path=str(p),
                    model_name=model_name,
                    detector_backend=detector_backend,
                    enforce_detection=False,
                    align=True,
                )
            except Exception:
                result = DeepFace.represent(
                    img_path=str(p),
                    model_name=model_name,
                    detector_backend="opencv",
                    enforce_detection=False,
                    align=True,
                )
            if isinstance(result, list):
                emb = result[0].get("embedding") if isinstance(result[0], dict) else result[0]
            else:
                emb = result.get("embedding")
            cache[str(p)] = [float(x) for x in emb]
            if i % 20 == 0:
                cache_path.write_text(json.dumps(cache), encoding="utf-8")
                print(f"embedded {i}/{len(missing)} new images")
        cache_path.write_text(json.dumps(cache), encoding="utf-8")
    return cache


def _evaluate_thresholds(distances, labels):
    distances = np.asarray(distances, dtype="float32")
    labels = np.asarray(labels, dtype="int32")
    grid = np.linspace(float(np.nanmin(distances)), float(np.nanmax(distances)), 350)
    best_f1 = {"threshold": None, "f1": -1, "accuracy": -1, "far": None, "frr": None}
    best_acc = {"threshold": None, "f1": -1, "accuracy": -1, "far": None, "frr": None}
    far_1pct = None
    far_5pct = None
    for thr in grid:
        pred = (distances <= thr).astype("int32")
        tp = int(((pred == 1) & (labels == 1)).sum())
        tn = int(((pred == 0) & (labels == 0)).sum())
        fp = int(((pred == 1) & (labels == 0)).sum())
        fn = int(((pred == 0) & (labels == 1)).sum())
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)
        acc = (tp + tn) / max(len(labels), 1)
        far = fp / max(fp + tn, 1)
        frr = fn / max(fn + tp, 1)
        row = {"threshold": float(thr), "f1": float(f1), "accuracy": float(acc), "far": float(far), "frr": float(frr)}
        if f1 > best_f1["f1"]:
            best_f1 = row
        if acc > best_acc["accuracy"]:
            best_acc = row
        if far <= 0.01 and (far_1pct is None or frr < far_1pct["frr"]):
            far_1pct = row
        if far <= 0.05 and (far_5pct is None or frr < far_5pct["frr"]):
            far_5pct = row
    return {
        "best_f1": best_f1,
        "best_accuracy": best_acc,
        "far_le_1pct": far_1pct,
        "far_le_5pct": far_5pct,
        "positive_distance_mean": float(distances[labels == 1].mean()) if (labels == 1).any() else None,
        "negative_distance_mean": float(distances[labels == 0].mean()) if (labels == 0).any() else None,
        "positive_distance_p95": float(np.quantile(distances[labels == 1], 0.95)) if (labels == 1).any() else None,
        "negative_distance_p05": float(np.quantile(distances[labels == 0], 0.05)) if (labels == 0).any() else None,
    }

if not RUN_CELEBA_THRESHOLD:
    print("CelebA calibration пропущена. Поставь RUN_CELEBA_THRESHOLD=True, если хочешь посчитать threshold сейчас.")
    print("Важно: нужен identity_CelebA.txt/list_identity_celeba.txt. list_eval_partition.txt не является identity-разметкой.")
else:
    root = Path(CELEBA_ROOT)
    if str(root).startswith("/content/drive") and not root.exists():
        from google.colab import drive
        drive.mount('/content/drive')
    root = Path(CELEBA_ROOT)
    out_path = root / "celeba_threshold_metrics.json"
    cache_path = root / f"celeba_embedding_cache_{CELEBA_MODEL_NAME}_{CELEBA_DETECTOR_BACKEND}.json"

    images_dir = _find_images_dir(root)
    identity_path = _find_identity_file(root)
    print("images_dir:", images_dir)
    print("identity_path:", identity_path)

    groups = _load_groups(identity_path, images_dir, CELEBA_MAX_IDENTITIES, CELEBA_RANDOM_SEED)
    pairs = _build_pairs(groups, CELEBA_POSITIVE_PAIRS, CELEBA_NEGATIVE_PAIRS, CELEBA_RANDOM_SEED)
    unique_paths = sorted({p for a, b, _ in pairs for p in (a, b)}, key=lambda p: str(p))
    print(f"identities={len(groups)}, pairs={len(pairs)}, unique_images={len(unique_paths)}")

    embeddings = _represent_cache(unique_paths, cache_path, CELEBA_MODEL_NAME, CELEBA_DETECTOR_BACKEND)
    distances, labels = [], []
    for a, b, label in pairs:
        if str(a) in embeddings and str(b) in embeddings:
            d = _cosine_distance(embeddings[str(a)], embeddings[str(b)])
            if math.isfinite(d):
                distances.append(d)
                labels.append(label)

    metrics = _evaluate_thresholds(distances, labels)
    result = {
        "ok": True,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "celeba_root": str(root),
        "images_dir": str(images_dir),
        "identity_path": str(identity_path),
        "model_name": CELEBA_MODEL_NAME,
        "detector_backend": CELEBA_DETECTOR_BACKEND,
        "identity_count_used": len(groups),
        "pair_count": len(labels),
        "positive_pairs": int(sum(labels)),
        "negative_pairs": int(len(labels) - sum(labels)),
        "metrics": metrics,
        "important_note": "Не ставь автоматически best_f1 как боевой ACCEPT threshold, если он слишком мягкий. Для demo оставляй conservative accept/review split.",
    }
    out_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(result, ensure_ascii=False, indent=2))
    print("Saved:", out_path)


## 2. Конфигурация проекта

Здесь я задаю основные пути, пороги и параметры подключения. Если Supabase не настроен, система остаётся рабочей за счёт `local_json` fallback. Это удобно для демонстрации: можно показать весь pipeline даже без активной БД, а при наличии Supabase те же функции будут сохранять профили и попытки проверки в таблицы.


In [ ]:
#@title 2. Конфигурация проекта безопасно: Colab Secrets / environment variables
import os
import subprocess
import sys
from pathlib import Path

USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_PROJECT_DIR = "mad_identity_verification" #@param {type:"string"}
OCR_DEVICE_MODE = "auto" #@param ["auto", "cpu", "gpu:0"]
OCR_QUALITY_PRESETS = True #@param {type:"boolean"}
OCR_WORKER_TIMEOUT_SEC = 420 #@param {type:"integer"}
FACE_MODEL_NAME = "Facenet512" #@param {type:"string"}
FACE_DETECTOR_BACKEND = "retinaface" #@param ["retinaface", "opencv"]
FACE_ACCEPT_THRESHOLD = 0.32 #@param {type:"number"}
FACE_REVIEW_THRESHOLD = 0.43 #@param {type:"number"}
DATA_ACCEPT_THRESHOLD = 0.80 #@param {type:"number"}
DATA_REVIEW_THRESHOLD = 0.55 #@param {type:"number"}

# Supabase credentials are intentionally not hardcoded.
# In Colab open the key icon (Secrets), add SUPABASE_URL and SUPABASE_KEY,
# then enable notebook access for both secrets.
def get_secret(name: str, default: str = "") -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return str(value).strip()
    except Exception:
        pass
    return os.getenv(name, default).strip()

SUPABASE_URL = get_secret("SUPABASE_URL")
SUPABASE_KEY = get_secret("SUPABASE_KEY")

os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")
os.environ.setdefault("MAD_AUTO_INSTALL_DEEPFACE", "1")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive') / DRIVE_PROJECT_DIR
else:
    BASE_DIR = Path('/content') / DRIVE_PROJECT_DIR

BASE_DIR.mkdir(parents=True, exist_ok=True)

def has_gpu_runtime():
    return subprocess.run(["bash", "-lc", "nvidia-smi >/dev/null 2>&1"]).returncode == 0

if OCR_DEVICE_MODE == "auto":
    try:
        import paddle
        OCR_DEVICE = "gpu:0" if has_gpu_runtime() and paddle.is_compiled_with_cuda() else "cpu"
    except Exception:
        OCR_DEVICE = "gpu:0" if has_gpu_runtime() else "cpu"
else:
    OCR_DEVICE = OCR_DEVICE_MODE

os.environ["MAD_BASE_DIR"] = str(BASE_DIR)
os.environ["APP_STORAGE_DIR"] = str(BASE_DIR / "storage")
os.environ["APP_ARTIFACTS_DIR"] = str(BASE_DIR / "artifacts")
os.environ["SUPABASE_URL"] = SUPABASE_URL
os.environ["SUPABASE_KEY"] = SUPABASE_KEY
os.environ["OCR_DEVICE"] = OCR_DEVICE
os.environ["OCR_QUALITY_PRESETS"] = "1" if OCR_QUALITY_PRESETS else "0"
os.environ["OCR_WORKER_TIMEOUT_SEC"] = str(OCR_WORKER_TIMEOUT_SEC)
os.environ["FACE_MODEL_NAME"] = FACE_MODEL_NAME
os.environ["FACE_DETECTOR_BACKEND"] = FACE_DETECTOR_BACKEND
os.environ["FACE_ACCEPT_THRESHOLD"] = str(FACE_ACCEPT_THRESHOLD)
os.environ["FACE_REVIEW_THRESHOLD"] = str(FACE_REVIEW_THRESHOLD)
os.environ["DATA_ACCEPT_THRESHOLD"] = str(DATA_ACCEPT_THRESHOLD)
os.environ["DATA_REVIEW_THRESHOLD"] = str(DATA_REVIEW_THRESHOLD)

print("BASE_DIR:", BASE_DIR)
print("OCR_DEVICE:", OCR_DEVICE)
print("Supabase configured:", bool(SUPABASE_URL and SUPABASE_KEY))
print("SUPABASE_URL source:", "Colab Secret / env" if SUPABASE_URL else "empty; local JSON fallback will be used")

# v4 PaddleOCR runtime defaults
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
os.environ["OCR_WORKER_TIMEOUT_SEC"] = "900"
os.environ["OCR_FAST_MODE"] = "1"
os.environ["OCR_QUALITY_PRESETS"] = "0"
os.environ["PADDLE_DET_LIMIT_TYPE"] = "max"
os.environ["PADDLE_DET_LIMIT_SIDE_LEN"] = "1600"
os.environ["PADDLE_TEXTLINE_ORIENTATION"] = "false"
os.environ["PADDLE_USE_DOC_UNWARPING"] = "false"
os.environ["PADDLE_DET_MODEL"] = "PP-OCRv5_mobile_det"
os.environ["PADDLE_REC_MODEL"] = "cyrillic_PP-OCRv5_mobile_rec"


## 1.7. Веса DeepFace

Перед запуском интерфейса я заранее скачиваю веса `Facenet512` и `RetinaFace`. Это вынесено в отдельный шаг, чтобы первая регистрация пользователя не падала из-за сетевого скачивания весов. Кэш хранится локально в Colab, а не в Google Drive, потому что Drive-mount иногда отваливается и даёт ошибку `Transport endpoint is not connected`.


In [ ]:
#@title 1.7. Предзагрузка весов DeepFace без зависимости от Google Drive
# Я заранее скачиваю веса Facenet512 и RetinaFace в локальные папки Colab.
# Это делает регистрацию стабильнее: DeepFace не пытается скачать веса во время нажатия кнопки в Gradio.

import os
import sys
import shutil
import subprocess
from pathlib import Path

# Важно:
# DeepFace сам добавляет ".deepface/weights" к DEEPFACE_HOME.
# Поэтому DEEPFACE_HOME должен быть "/root", а не "/root/.deepface".
os.environ["DEEPFACE_HOME"] = "/root"

# Не используем Google Drive в этой ячейке.
# Именно Drive чаще всего даёт:
# OSError: [Errno 107] Transport endpoint is not connected
USE_GOOGLE_DRIVE_CACHE = False

MAIN_WEIGHTS_DIR = Path("/root/.deepface/weights")

# Совместимость с ошибочным путём, который иногда появляется,
# если DEEPFACE_HOME ранее был задан как /root/.deepface.
LEGACY_WEIGHTS_DIR = Path("/root/.deepface/.deepface/weights")

# Локальный cache в /content. Не переживает полный reset runtime,
# но не зависит от Google Drive.
LOCAL_CACHE_DIR = Path("/content/deepface_cache/weights")

WEIGHT_DIRS = [
    MAIN_WEIGHTS_DIR,
    LEGACY_WEIGHTS_DIR,
    LOCAL_CACHE_DIR,
]

if USE_GOOGLE_DRIVE_CACHE:
    # По умолчанию выключено.
    # Включать только если Drive точно стабилен.
    WEIGHT_DIRS.append(Path("/content/drive/MyDrive/mad_identity_verification/deepface_cache/weights"))


ASSETS = {
    "facenet512_weights.h5": {
        "min_bytes": 80_000_000,
        "urls": [
            "https://github.com/serengil/deepface_models/releases/download/v1.0/facenet512_weights.h5",
            "https://huggingface.co/WePrompt/deepface-weights/resolve/main/facenet512_weights.h5",
        ],
    },
    "retinaface.h5": {
        "min_bytes": 80_000_000,
        "urls": [
            "https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5",
        ],
    },
}


def safe_mkdir(path: Path) -> bool:
    try:
        path.mkdir(parents=True, exist_ok=True)
        return True
    except OSError as exc:
        print(f"SKIP mkdir, path is not usable: {path} | {repr(exc)}")
        return False


def safe_is_valid_file(path: Path, min_bytes: int) -> bool:
    try:
        return path.exists() and path.is_file() and path.stat().st_size >= min_bytes
    except OSError as exc:
        print(f"SKIP invalid path: {path} | {repr(exc)}")
        return False


def safe_unlink(path: Path) -> None:
    try:
        if path.exists():
            path.unlink()
    except OSError as exc:
        print(f"SKIP unlink, path is not usable: {path} | {repr(exc)}")


def prepare_dirs():
    usable = []
    for d in WEIGHT_DIRS:
        if safe_mkdir(d):
            usable.append(d)

    if not usable:
        raise RuntimeError("No usable DeepFace weight directories.")

    return usable


USABLE_WEIGHT_DIRS = prepare_dirs()

print("Usable DeepFace weight dirs:")
for d in USABLE_WEIGHT_DIRS:
    print(" -", d)


def find_existing_valid_file(filename: str, min_bytes: int):
    for d in USABLE_WEIGHT_DIRS:
        p = d / filename
        if safe_is_valid_file(p, min_bytes):
            return p
    return None


def copy_to_all_dirs(source: Path, filename: str, min_bytes: int):
    if not safe_is_valid_file(source, min_bytes):
        raise RuntimeError(f"Source file is invalid: {source}")

    for d in USABLE_WEIGHT_DIRS:
        dst = d / filename

        try:
            if dst.resolve() == source.resolve():
                continue
        except Exception:
            pass

        if safe_is_valid_file(dst, min_bytes):
            continue

        try:
            safe_unlink(dst)
            shutil.copy2(source, dst)
            print(f"Copied {filename} -> {dst}")
        except OSError as exc:
            print(f"SKIP copy to unusable path: {dst} | {repr(exc)}")

    print(f"OK: {filename} is available in local DeepFace folders.")


def download_with_shell(url: str, destination: Path, min_bytes: int) -> bool:
    destination.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(destination) + ".tmp")

    safe_unlink(tmp)

    commands = [
        [
            "wget",
            "--tries=3",
            "--timeout=120",
            "--read-timeout=120",
            "-O",
            str(tmp),
            url,
        ],
        [
            "curl",
            "-L",
            "--retry",
            "3",
            "--connect-timeout",
            "60",
            "--max-time",
            "900",
            "-o",
            str(tmp),
            url,
        ],
    ]

    for cmd in commands:
        print("\nTrying:", cmd[0], url)
        result = subprocess.run(cmd, text=True)

        if result.returncode == 0 and safe_is_valid_file(tmp, min_bytes):
            safe_unlink(destination)
            tmp.rename(destination)
            size_mb = destination.stat().st_size / 1024 / 1024
            print(f"Downloaded: {destination} ({size_mb:.1f} MB)")
            return True

        if tmp.exists():
            try:
                size_mb = tmp.stat().st_size / 1024 / 1024
                print(f"Invalid/partial file size: {size_mb:.1f} MB")
            except Exception:
                pass
            safe_unlink(tmp)

    return False


for filename, spec in ASSETS.items():
    min_bytes = spec["min_bytes"]

    existing = find_existing_valid_file(filename, min_bytes)
    if existing is not None:
        print(f"\nFound cached {filename}: {existing}")
        copy_to_all_dirs(existing, filename, min_bytes)
        continue

    # Скачиваем сначала в локальный /content cache.
    target = LOCAL_CACHE_DIR / filename

    downloaded = False
    for url in spec["urls"]:
        if download_with_shell(url, target, min_bytes):
            downloaded = True
            break

    if not downloaded:
        raise RuntimeError(
            f"Could not download {filename}. "
            f"Manual option: upload it to {LOCAL_CACHE_DIR}"
        )

    copy_to_all_dirs(target, filename, min_bytes)


# Удаляем частично импортированные DeepFace/RetinaFace модули,
# чтобы после скачивания весов они перечитались корректно.
for module_name in list(sys.modules.keys()):
    if (
        module_name == "deepface"
        or module_name.startswith("deepface.")
        or module_name == "retinaface"
        or module_name.startswith("retinaface.")
    ):
        sys.modules.pop(module_name, None)


print("\nChecking DeepFace / Facenet512 model loading...")

from deepface import DeepFace

try:
    model = DeepFace.build_model("Facenet512")
except TypeError:
    model = DeepFace.build_model(model_name="Facenet512")

print("DeepFace Facenet512 loaded successfully.")

print("\nWeights status:")
for d in USABLE_WEIGHT_DIRS:
    print("DIR:", d)
    for filename in ASSETS:
        p = d / filename
        if safe_is_valid_file(p, ASSETS[filename]["min_bytes"]):
            print("  OK ", filename, round(p.stat().st_size / 1024 / 1024, 1), "MB")
        else:
            print("  MISS", filename)

print("\nDeepFace weights are ready.")


## 3. Распаковка проекта и подготовка Python-файлов

В этой ячейке я распаковываю внешний ZIP проекта в рабочую папку `mad_identity_verification`. После распаковки автоматически проверяются и финализируются два файла:

- `mad_colab_pkg/paddle_ocr_worker.py` — отдельный OCR-worker для PaddleOCR;
- `training/train_text_classifier.py` — скрипт обучения моего text-line classifier.

Именно здесь применены уже проверенные compatibility-правки: удаление устаревшего аргумента `engine="paddle"` для PaddleOCR 3.3.3 и поддержка `--seed` в обучающем скрипте. Отдельной patch-ячейки больше нет.


In [ ]:
#@title 3. Распаковка проекта и автоматическая финализация файлов
# Проект берётся из отдельного ZIP-файла, поэтому notebook остаётся лёгким и не тормозит Colab.

import os
import shutil
import sys
import tempfile
import zipfile
from pathlib import Path

PROJECT_BUNDLE_VERSION = "v4_paddle_gpu_torch_repair_light_no_base64"
PROJECT_ZIP_NAMES = [
    "MAD_Identity_Verification_Runtime_v4_Final.zip",
    "mad_colab_v4_light_runtime.zip",
    "mad_colab_only_project_v4_paddle_gpu_fixed.zip",
]

# BASE_DIR обычно задаётся в ячейке 2. Если runtime перезапускался, восстановим безопасно.
BASE_DIR = Path(os.environ.get("MAD_BASE_DIR", "/content/drive/MyDrive/mad_identity_verification"))
BASE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["MAD_BASE_DIR"] = str(BASE_DIR)

print("BASE_DIR:", BASE_DIR)
print("Looking for project ZIP...")

search_dirs = [
    Path.cwd(),
    Path("/content"),
    Path("/content/drive/MyDrive"),
    BASE_DIR,
]

# Добавим похожие папки в Drive, если Drive подключен.
drive_root = Path("/content/drive/MyDrive")
if drive_root.exists():
    search_dirs.extend(sorted(drive_root.glob("mad_identity_verification*")))

candidates = []
for d in search_dirs:
    if not d.exists() or not d.is_dir():
        continue
    for name in PROJECT_ZIP_NAMES:
        p = d / name
        if p.exists():
            candidates.append(p)
    candidates.extend(sorted(d.glob("mad_colab_v4*runtime*.zip")))
    candidates.extend(sorted(d.glob("mad_colab_only_project_v4*.zip")))

# Убираем дубли.
unique = []
seen = set()
for p in candidates:
    try:
        rp = p.resolve()
    except Exception:
        rp = p
    if str(rp) not in seen:
        seen.add(str(rp))
        unique.append(rp)

project_zip = unique[0] if unique else None

if project_zip is None:
    print("\nZIP not found in /content or Google Drive.")
    print("Upload one of these files now:")
    for name in PROJECT_ZIP_NAMES:
        print(" -", name)

    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No ZIP uploaded.")

    uploaded_names = list(uploaded.keys())
    zip_names = [name for name in uploaded_names if name.lower().endswith(".zip")]
    if not zip_names:
        raise RuntimeError(f"Uploaded files do not include ZIP: {uploaded_names}")

    project_zip = Path(zip_names[0]).resolve()

    # Сохраним ZIP в Drive/BASE_DIR, чтобы в следующий раз не загружать руками.
    try:
        cached_zip = BASE_DIR / project_zip.name
        if project_zip.resolve() != cached_zip.resolve():
            shutil.copy2(project_zip, cached_zip)
            print("Cached uploaded ZIP to:", cached_zip)
    except Exception as exc:
        print("Could not cache ZIP, continuing:", repr(exc))

print("Using project ZIP:", project_zip)
print("ZIP size MB:", round(project_zip.stat().st_size / 1024 / 1024, 2))

extract_dir = Path(tempfile.mkdtemp(prefix="mad_project_extract_"))
with zipfile.ZipFile(project_zip, "r") as z:
    z.extractall(extract_dir)

# Поддерживаем оба формата архива:
# 1) mad_colab_v4_project/... внутри ZIP
# 2) файлы проекта прямо в корне ZIP
possible_roots = [
    extract_dir / "mad_colab_v4_project",
    extract_dir / "mad_colab_v3_project",
    extract_dir / "mad_colab_project",
    extract_dir,
]

src_root = None
for p in possible_roots:
    if (p / "mad_colab_pkg" / "colab_app.py").exists():
        src_root = p
        break

if src_root is None:
    all_files = [str(p.relative_to(extract_dir)) for p in extract_dir.rglob("*")][:80]
    raise RuntimeError("Could not find mad_colab_pkg/colab_app.py inside ZIP. First files:\n" + "\n".join(all_files))

print("Source root inside ZIP:", src_root)

# Не трогаем storage, чтобы не стереть local_store.json, uploads и старые OCR jobs.
# Остальные директории перезаписываем, чтобы точно не остались старые .py файлы.
copy_items = [
    "mad_colab_pkg",
    "training",
    "sql",
    "examples",
    "data",
    "automation",
    "artifacts",
    "README_COLAB_ONLY.md",
]

for name in copy_items:
    src = src_root / name
    if not src.exists():
        continue
    dst = BASE_DIR / name
    if dst.exists():
        if dst.is_dir():
            shutil.rmtree(dst)
        else:
            dst.unlink()
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print("Copied:", name)

# Создаём нужные runtime-папки.
for rel in [
    "storage",
    "storage/uploads",
    "storage/uploads/reference",
    "storage/uploads/passport_ocr",
    "storage/uploads/passport_verify",
    "storage/uploads/selfie",
    "storage/ocr_jobs",
    "artifacts",
]:
    (BASE_DIR / rel).mkdir(parents=True, exist_ok=True)

(BASE_DIR / "PROJECT_VERSION.txt").write_text(f"{PROJECT_BUNDLE_VERSION}\n", encoding="utf-8")

# Обновляем sys.path и сбрасываем старые импорты проекта.
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

for module_name in list(sys.modules.keys()):
    if module_name == "mad_colab_pkg" or module_name.startswith("mad_colab_pkg."):
        sys.modules.pop(module_name, None)

print("\nProject unpacked without embedded base64.")
print("PROJECT_VERSION:", (BASE_DIR / "PROJECT_VERSION.txt").read_text(encoding="utf-8").strip())
print("face_service.py exists:", (BASE_DIR / "mad_colab_pkg" / "face_service.py").exists())
print("paddle_ocr_worker.py exists:", (BASE_DIR / "mad_colab_pkg" / "paddle_ocr_worker.py").exists())
print("train_text_classifier.py exists:", (BASE_DIR / "training" / "train_text_classifier.py").exists())


# ------------------------------------------------------------
# Project compatibility finalization
# После распаковки я привожу два файла к текущей среде Colab:
# - paddle_ocr_worker.py: совместимость с paddleocr==3.3.3;
# - train_text_classifier.py: поддержка параметра --seed для воспроизводимого обучения.
# Логика взята из уже проверенных рабочих фиксов и применяется автоматически.
# ------------------------------------------------------------

import os
import sys
from pathlib import Path

def find_mad_base_dir() -> Path:
    candidates = []

    env_value = os.environ.get("MAD_BASE_DIR")
    if env_value:
        candidates.append(Path(env_value))

    global_value = globals().get("BASE_DIR")
    if global_value:
        candidates.append(Path(global_value))

    candidates.extend([
        Path("/content/drive/MyDrive/mad_identity_verification"),
        Path("/content/mad_identity_verification"),
    ])

    drive_root = Path("/content/drive/MyDrive")
    if drive_root.exists():
        candidates.extend(sorted(drive_root.glob("mad_identity_verification*")))

    seen = set()
    unique = []

    for p in candidates:
        try:
            rp = p.resolve()
        except Exception:
            rp = p

        if str(rp) not in seen:
            seen.add(str(rp))
            unique.append(rp)

    for p in unique:
        if (
            (p / "mad_colab_pkg" / "paddle_ocr_worker.py").exists()
            and (p / "training" / "train_text_classifier.py").exists()
        ):
            return p

    checked = "\n".join(f" - {p}" for p in unique)
    raise RuntimeError(
        "Не удалось найти папку проекта mad_identity_verification.\n"
        "Сначала выполни ячейку 3 с распаковкой проекта.\n\n"
        f"Проверенные пути:\n{checked}"
    )

BASE_DIR = find_mad_base_dir()
os.environ["MAD_BASE_DIR"] = str(BASE_DIR)

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

print("BASE_DIR:", BASE_DIR)

# ------------------------------------------------------------
# 1. Patch paddle_ocr_worker.py
# ------------------------------------------------------------

worker_path = BASE_DIR / "mad_colab_pkg" / "paddle_ocr_worker.py"
worker_text = worker_path.read_text(encoding="utf-8")

original_worker_text = worker_text

# Remove old PaddleOCR 3.x-incompatible argument.
worker_text = worker_text.replace('        engine="paddle",\n', "")

# Add robust retry: if future PaddleOCR rejects some kwarg, remove it and retry.
old_return_block = '''    print(json.dumps({"stage": "build_ocr", "kwargs": kwargs}, ensure_ascii=False), flush=True)
    return PaddleOCR(**kwargs)
'''

new_return_block = '''    print(json.dumps({"stage": "build_ocr", "kwargs": kwargs}, ensure_ascii=False), flush=True)

    # PaddleOCR 3.x has changed accepted kwargs across minor versions.
    # If a kwarg is rejected as "Unknown argument: X", drop it and retry.
    while True:
        try:
            return PaddleOCR(**kwargs)
        except ValueError as exc:
            msg = str(exc)
            prefix = "Unknown argument: "
            if prefix in msg:
                bad_arg = msg.split(prefix, 1)[1].strip().strip("'").strip('"')
                if bad_arg in kwargs:
                    kwargs.pop(bad_arg, None)
                    print(
                        json.dumps(
                            {
                                "stage": "build_ocr_retry_without_unknown_arg",
                                "removed": bad_arg,
                                "kwargs": kwargs,
                            },
                            ensure_ascii=False,
                        ),
                        flush=True,
                    )
                    continue
            raise
'''

if old_return_block in worker_text:
    worker_text = worker_text.replace(old_return_block, new_return_block)
else:
    print("WARNING: expected return block not found in worker; only removed engine line.")

if worker_text != original_worker_text:
    worker_path.write_text(worker_text, encoding="utf-8")
    print("Patched worker:", worker_path)
else:
    print("Worker already patched or no changes needed:", worker_path)

# Quick verification.
patched_worker_text = worker_path.read_text(encoding="utf-8")
print("worker contains engine=\"paddle\":", 'engine="paddle"' in patched_worker_text)
print("worker has retry_unknown_arg:", "build_ocr_retry_without_unknown_arg" in patched_worker_text)

# ------------------------------------------------------------
# 2. Patch training/train_text_classifier.py
# ------------------------------------------------------------

train_path = BASE_DIR / "training" / "train_text_classifier.py"
train_text = train_path.read_text(encoding="utf-8")

original_train_text = train_text

# Add global RANDOM_SEED inside main() if not present.
if "def main():\n    global RANDOM_SEED\n" not in train_text:
    train_text = train_text.replace(
        "def main():\n    parser = argparse.ArgumentParser()",
        "def main():\n    global RANDOM_SEED\n    parser = argparse.ArgumentParser()",
    )

# Add --seed argument if not present.
if 'parser.add_argument("--seed"' not in train_text:
    train_text = train_text.replace(
        '    parser.add_argument("--dataset-out", default=None)\n    args = parser.parse_args()\n',
        '    parser.add_argument("--dataset-out", default=None)\n'
        '    parser.add_argument("--seed", type=int, default=RANDOM_SEED)\n'
        '    args = parser.parse_args()\n\n'
        '    RANDOM_SEED = int(args.seed)\n'
        '    random.seed(RANDOM_SEED)\n'
        '    np.random.seed(RANDOM_SEED)\n',
    )

# Make metrics explicitly record the seed.
if '"seed": int(RANDOM_SEED),' not in train_text:
    train_text = train_text.replace(
        '        "input_csv_sha256": sha256_file(csv_path),\n',
        '        "input_csv_sha256": sha256_file(csv_path),\n'
        '        "seed": int(RANDOM_SEED),\n',
    )

if train_text != original_train_text:
    train_path.write_text(train_text, encoding="utf-8")
    print("Patched training script:", train_path)
else:
    print("Training script already patched or no changes needed:", train_path)

# Quick verification.
patched_train_text = train_path.read_text(encoding="utf-8")
print("train script accepts --seed:", 'parser.add_argument("--seed"' in patched_train_text)
print("train script records seed:", '"seed": int(RANDOM_SEED),' in patched_train_text)

# ------------------------------------------------------------
# 3. Clear imported project modules
# ------------------------------------------------------------

for name in list(sys.modules.keys()):
    if name == "mad_colab_pkg" or name.startswith("mad_colab_pkg."):
        sys.modules.pop(name, None)

print("\nProject compatibility finalization finished.")


## 4. Проверка окружения и warmup OCR

Здесь я проверяю, что импортируются `torch`, `paddle` и `paddleocr`, что Paddle видит CUDA, а parser корректно собирает паспортные поля на тестовом OCR-примере. Затем я запускаю `mad_colab_pkg/paddle_ocr_worker.py` на маленьком warmup-изображении. Это позволяет скачать/инициализировать OCR-модели до открытия Gradio и увидеть проблему сразу в notebook, а не после нажатия кнопки в интерфейсе.


In [ ]:
#@title 4. Проверка окружения, parser sanity-check и warmup PaddleOCR worker
import json
import os
import subprocess
import sys
from pathlib import Path

BASE_DIR = Path(os.environ.get("MAD_BASE_DIR", "/content/mad_identity_verification"))
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

print("BASE_DIR:", BASE_DIR)
print("PROJECT_VERSION:", (BASE_DIR / "PROJECT_VERSION.txt").read_text(encoding="utf-8").strip() if (BASE_DIR / "PROJECT_VERSION.txt").exists() else "missing")

print("\n=== Runtime import checks ===")
for name, code in {
    "torch": "import torch; print(torch.__version__, torch.cuda.is_available() if hasattr(torch, 'cuda') else False)",
    "paddle": "import paddle; print(paddle.__version__, paddle.is_compiled_with_cuda())",
    "paddleocr": "import os; os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK']='True'; from paddleocr import PaddleOCR; print('ok')",
}.items():
    print("---", name, "---")
    r = subprocess.run([sys.executable, "-c", code], text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-4000:])

try:
    import paddle
    os.environ["OCR_DEVICE"] = "gpu:0" if paddle.is_compiled_with_cuda() else "cpu"
except Exception:
    os.environ["OCR_DEVICE"] = "cpu"
print("OCR_DEVICE:", os.environ.get("OCR_DEVICE"))

from mad_colab_pkg.passport_parser import parse_passport_data
sample_path = BASE_DIR / "examples" / "all_ocr_items.json"
if sample_path.exists():
    parsed = parse_passport_data(json.loads(sample_path.read_text(encoding="utf-8")))
    print("\nParser sample passport_data:")
    print(json.dumps(parsed.get("passport_data", {}), ensure_ascii=False, indent=2))

# Small PaddleOCR worker warmup. It downloads/loads OCR models before Gradio.
RUN_PADDLEOCR_WARMUP = True
if RUN_PADDLEOCR_WARMUP:
    from PIL import Image, ImageDraw
    warmup_dir = BASE_DIR / "storage" / "warmup"
    warmup_dir.mkdir(parents=True, exist_ok=True)
    img_path = warmup_dir / "warmup.jpg"
    img = Image.new("RGB", (900, 220), "white")
    d = ImageDraw.Draw(img)
    d.text((40, 70), "TEST 123 11.01.2018", fill="black")
    img.save(img_path, quality=95)
    manifest_path = warmup_dir / "manifest.json"
    output_path = warmup_dir / "paddle_ocr_output.json"
    manifest_path.write_text(json.dumps([{"tag": "warmup", "path": str(img_path)}], ensure_ascii=False), encoding="utf-8")

    env = os.environ.copy()
    env.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")
    env.setdefault("PADDLE_DET_LIMIT_TYPE", "max")
    env.setdefault("PADDLE_DET_LIMIT_SIDE_LEN", "1600")
    env.setdefault("PADDLE_TEXTLINE_ORIENTATION", "false")
    env.setdefault("PADDLE_DET_MODEL", "PP-OCRv5_mobile_det")
    env.setdefault("PADDLE_REC_MODEL", "cyrillic_PP-OCRv5_mobile_rec")
    env.setdefault("OCR_DEVICE", os.environ.get("OCR_DEVICE", "cpu"))

    worker = BASE_DIR / "mad_colab_pkg" / "paddle_ocr_worker.py"
    cmd = [sys.executable, str(worker), "--manifest", str(manifest_path), "--output", str(output_path)]
    print("\n=== PaddleOCR worker warmup ===")
    proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, env=env, timeout=900)
    print("returncode:", proc.returncode)
    print("stdout:\n", proc.stdout[-4000:])
    print("stderr:\n", proc.stderr[-4000:])
    if output_path.exists():
        print("output JSON:")
        print(output_path.read_text(encoding="utf-8")[:4000])
    else:
        print("output JSON was not created")


## 5. Обучение собственного text-line classifier

Эта ячейка обучает модель классификации OCR-строк на `data/passport-2000.csv` и синтетически зашумленных примерах. Модель не заменяет parser, а помогает определить тип строки: ФИО, дата, номер, код подразделения, MRZ или шум. Финальный extraction всё равно использует ROI, regex, MRZ, validators и consensus, потому что реальный OCR может ошибаться.

Используемый файл: `training/train_text_classifier.py`.


In [ ]:
#@title 5. Обучение собственного text-line classifier внутри Colab
import json
import os
import subprocess
import sys
from pathlib import Path

BASE_DIR = Path(os.environ.get("MAD_BASE_DIR", "/content/drive/MyDrive/mad_identity_verification"))
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

csv_path = BASE_DIR / "data" / "passport-2000.csv"
out_path = BASE_DIR / "artifacts" / "text_line_classifier.joblib"
metrics_path = BASE_DIR / "artifacts" / "text_line_classifier_metrics.json"
dataset_path = BASE_DIR / "artifacts" / "text_line_classifier_dataset.csv"

FORCE_RETRAIN_TEXT_CLASSIFIER = True #@param {type:"boolean"}

if not csv_path.exists():
    raise FileNotFoundError(f"passport-2000.csv not found: {csv_path}. Run project unpack cell first.")

if FORCE_RETRAIN_TEXT_CLASSIFIER:
    for p in [out_path, metrics_path, dataset_path]:
        if p.exists():
            p.unlink()
            print("Deleted old artifact:", p)

script = BASE_DIR / "training" / "train_text_classifier.py"
cmd = [sys.executable, str(script), "--csv", str(csv_path), "--out", str(out_path), "--metrics", str(metrics_path), "--dataset-out", str(dataset_path), "--seed", "42"]
print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, cwd=str(BASE_DIR))
print("STDOUT:\n", proc.stdout[-8000:])
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr[-8000:])
    raise RuntimeError("Text classifier training failed")

print("\nMetrics:")
print(metrics_path.read_text(encoding="utf-8"))


## 6. SQL-схема Supabase

Эта ячейка выводит SQL для двух таблиц: `identity_profiles` и `verification_attempts`. Для учебной демонстрации можно отключить RLS, чтобы не получить ошибку записи профиля. В production-режиме вместо этого нужны полноценные политики доступа.


In [ ]:
#@title 6. Supabase SQL schema
import os
from pathlib import Path

BASE_DIR = Path(os.environ.get("MAD_BASE_DIR", "/content/mad_identity_verification"))
schema_path = BASE_DIR / "sql" / "supabase_schema.sql"
print(schema_path.read_text(encoding="utf-8"))


## Что именно сверяется с Supabase

Вкладка **OCR паспорта** только распознаёт документ и создаёт debug-файлы. Вкладка **Полная проверка** делает OCR, ищет лучший профиль в Supabase или local JSON и считает `data_match_score` по серии, номеру, дате рождения и ФИО. Затем добавляется face verification и формируется `ACCEPT / REVIEW / REJECT`.


## 7. Запуск Gradio-интерфейса

Здесь я запускаю пользовательский интерфейс. Gradio не является отдельным backend-сервером: кнопки вызывают Python-функции из `mad_colab_pkg/colab_app.py`, а основная логика находится в `mad_colab_pkg/colab_pipeline.py`.

В интерфейсе есть регистрация, OCR паспорта, полная проверка и экспорт runtime. Полная проверка делает OCR, ищет лучший профиль в Supabase/local JSON, считает совпадение паспортных данных и добавляет face verification.


In [ ]:
#@title 7. Запуск web-интерфейса Gradio
import os
import sys
from pathlib import Path

BASE_DIR = Path(os.environ.get("MAD_BASE_DIR", "/content/mad_identity_verification"))
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

# Ensure OCR env survives before Gradio callbacks.
try:
    import paddle
    os.environ["OCR_DEVICE"] = "gpu:0" if paddle.is_compiled_with_cuda() else "cpu"
except Exception:
    os.environ.setdefault("OCR_DEVICE", "cpu")

os.environ.setdefault("OCR_WORKER_TIMEOUT_SEC", "900")
os.environ.setdefault("OCR_FAST_MODE", "1")
os.environ.setdefault("OCR_QUALITY_PRESETS", "0")
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")
os.environ.setdefault("PADDLE_DET_LIMIT_TYPE", "max")
os.environ.setdefault("PADDLE_DET_LIMIT_SIDE_LEN", "1600")
os.environ.setdefault("PADDLE_TEXTLINE_ORIENTATION", "false")
os.environ.setdefault("PADDLE_USE_DOC_UNWARPING", "false")
os.environ.setdefault("PADDLE_DET_MODEL", "PP-OCRv5_mobile_det")
os.environ.setdefault("PADDLE_REC_MODEL", "cyrillic_PP-OCRv5_mobile_rec")

from mad_colab_pkg.colab_app import launch_gradio

demo = launch_gradio(share=True)


## 8. Экспорт результатов

Эта ячейка собирает runtime-архив с безопасным набором файлов проекта. Чувствительные runtime-данные, uploads, local stores и debug-артефакты исключаются из экспорта.


In [ ]:
#@title 8. Экспорт результатов runtime
import os
import sys
from pathlib import Path
from google.colab import files

BASE_DIR = Path(os.environ.get("MAD_BASE_DIR", "/content/drive/MyDrive/mad_identity_verification"))
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from mad_colab_pkg.colab_pipeline import make_runtime_zip
zip_path = make_runtime_zip()
print("ZIP created:", zip_path)
files.download(zip_path)
